# Project 89 — Local Voice Notes Organizer
## Raw Notes → Categorize → Link → Search

**Stack:** LangChain · Ollama · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb pydantic

## Step 1 — Simulated Voice Notes

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

voice_notes = [
    {"id": "vn_001", "timestamp": "2025-03-01 09:15",
     "text": "Idea for the app — we should add a dark mode toggle in settings. "
             "Users have been requesting it. Check the GitHub issues for upvotes."},
    {"id": "vn_002", "timestamp": "2025-03-01 14:30",
     "text": "Meeting with Sarah went well. She approved the Q2 budget — we got "
             "an extra $50K for cloud infrastructure. Need to update the proposal."},
    {"id": "vn_003", "timestamp": "2025-03-02 08:00",
     "text": "Remember to buy groceries — milk, eggs, bread, chicken. Also pick up "
             "dry cleaning before 6pm."},
    {"id": "vn_004", "timestamp": "2025-03-02 11:45",
     "text": "The caching bug is in the Redis layer. When TTL expires during a write "
             "operation, we get stale data. Need to implement write-through cache."},
    {"id": "vn_005", "timestamp": "2025-03-03 16:00",
     "text": "Book recommendation from Dave — 'Designing Data-Intensive Applications' "
             "by Martin Kleppmann. Good for understanding distributed systems."},
    {"id": "vn_006", "timestamp": "2025-03-03 19:30",
     "text": "Workout plan for this week: Monday legs, Wednesday upper body, "
             "Friday cardio and core. Remember to stretch before each session."},
    {"id": "vn_007", "timestamp": "2025-03-04 10:00",
     "text": "API rate limiting should be 100 requests per minute per user. "
             "Use token bucket algorithm. Need to discuss with the team tomorrow."},
]
print(f"Voice notes: {len(voice_notes)}")

## Step 2 — Auto-Categorize & Extract Metadata

In [ ]:
class NoteMetadata(BaseModel):
    category: str = Field(description="work, personal, idea, bug, meeting, reference")
    priority: str = Field(description="high, medium, low")
    has_action_item: bool
    action_item: str = ""
    related_people: list[str]
    tags: list[str]

classifier = llm.with_structured_output(NoteMetadata)

enriched = []
for note in voice_notes:
    meta = classifier.invoke(f"Categorize this voice note:\n{note['text']}")
    enriched.append({**note, "metadata": meta})
    print(f"  {note['id']} [{meta.category}] P:{meta.priority} "
          f"{'📌' if meta.has_action_item else '  '} tags={meta.tags[:3]}")

## Step 3 — Store in Vector DB for Search

In [ ]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection("voice_notes")

for note in enriched:
    meta = note["metadata"]
    collection.add(
        ids=[note["id"]],
        documents=[note["text"]],
        metadatas=[{
            "timestamp": note["timestamp"],
            "category": meta.category,
            "priority": meta.priority,
            "has_action": str(meta.has_action_item),
        }],
    )

print(f"✓ Indexed {collection.count()} notes in ChromaDB")

## Step 4 — Semantic Search & Linking

In [ ]:
queries = [
    "What bugs do I need to fix?",
    "What meetings happened this week?",
    "Any ideas for the app?",
    "What tasks need to be done today?",
]

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on the voice notes. Be specific and reference the notes."),
    ("human", "Voice notes:\n{notes}\n\nQuestion: {question}\nAnswer:")
])
qa_chain = qa_prompt | llm | StrOutputParser()

for query in queries:
    results = collection.query(query_texts=[query], n_results=3)
    context = "\n".join(f"- {doc}" for doc in results["documents"][0])
    answer = qa_chain.invoke({"notes": context, "question": query})
    print(f"\nQ: {query}")
    print(f"A: {answer[:150]}")

## Step 5 — Daily Digest

In [ ]:
digest_prompt = ChatPromptTemplate.from_messages([
    ("system", "Create a daily digest from these voice notes. Group by category. "
     "Highlight action items. Be concise."),
    ("human", "Notes:\n{notes}\n\nDaily Digest:")
])
digest_chain = digest_prompt | llm | StrOutputParser()

all_notes = "\n".join(f"[{n['timestamp']}] {n['text']}" for n in voice_notes)
digest = digest_chain.invoke({"notes": all_notes})
print("DAILY DIGEST")
print("=" * 50)
print(digest[:500])

## What You Learned
- **Voice note auto-categorization** with structured output
- **Vector search** for semantic note retrieval
- **Cross-note linking** via embeddings
- **Daily digest generation** from unstructured notes